# 🎬 KNN Movie Recommendation System
Built using Nearest Neighbors (KNN)

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors


## 2. Load Dataset

In [3]:
df = pd.read_csv("Data for repository.csv")
df.head()

,Movie_Name,Release_Period,Whether_Remake,Whether_Franchise,Genre,New_Actor,New_Director,New_Music_Director,Lead_Star,Director,Music_Director,Number_of_Screens,Revenue(INR),Budget(INR)
0,Golden Boys,Normal,No,No,suspense,Yes,No,No,Jeet Goswami,Ravi Varma,Baba Jagirdar,5,5000000,85000
1,Kaccha Limboo,Holiday,No,No,drama,Yes,No,Yes,Karan Bhanushali,Sagar Ballary,Amardeep Nijjer,75,15000000,825000
2,Not A Love Story,Holiday,No,No,thriller,No,No,No,Mahie Gill,Ram Gopal Verma,Sandeep Chowta,525,75000000,56700000
3,Qaidi Band,Holiday,No,No,drama,Yes,No,No,Aadar Jain,Habib Faisal,Amit Trivedi,800,210000000,4500000
4,Chaatwali,Holiday,No,No,adult,Yes,Yes,Yes,Aadil Khan,Aadil Khan,Babloo Ustad,1,1000000,1075000


## 3. Explore Dataset

In [4]:
print(df.shape)
df.info()
df.describe(include="all")
df.isnull().sum()

(1698, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Movie_Name          1698 non-null   str  
 1   Release_Period      1698 non-null   str  
 2   Whether_Remake      1698 non-null   str  
 3   Whether_Franchise   1698 non-null   str  
 4   Genre               1698 non-null   str  
 5   New_Actor           1698 non-null   str  
 6   New_Director        1698 non-null   str  
 7   New_Music_Director  1698 non-null   str  
 8   Lead_Star           1698 non-null   str  
 9   Director            1698 non-null   str  
 10  Music_Director      1698 non-null   str  
 11  Number_of_Screens   1698 non-null   int64
 12  Revenue(INR)        1698 non-null   int64
 13  Budget(INR)         1698 non-null   int64
dtypes: int64(3), str(11)
memory usage: 185.8 KB


Movie_Name            0
Release_Period        0
Whether_Remake        0
Whether_Franchise     0
Genre                 0
New_Actor             0
New_Director          0
New_Music_Director    0
Lead_Star             0
Director              0
Music_Director        0
Number_of_Screens     0
Revenue(INR)          0
Budget(INR)           0
dtype: int64

## 4. Data Cleaning

In [5]:
df = df.drop_duplicates()

categorical_cols = [
    'Release_Period','Whether_Remake','Whether_Franchise',
    'Genre','New_Actor','New_Director','New_Music_Director',
    'Lead_Star','Director','Music_Director'
]

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

numeric_cols = ['Number_of_Screens','Revenue(INR)','Budget(INR)']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print(df.isnull().sum())


Movie_Name            0
Release_Period        0
Whether_Remake        0
Whether_Franchise     0
Genre                 0
New_Actor             0
New_Director          0
New_Music_Director    0
Lead_Star             0
Director              0
Music_Director        0
Number_of_Screens     0
Revenue(INR)          0
Budget(INR)           0
dtype: int64


## 5. Label Encoding

In [6]:
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le


## 6. Feature Matrix

In [7]:
feature_cols = categorical_cols + numeric_cols

X = df[feature_cols]
movie_names = df["Movie_Name"]

X.head()

,Release_Period,Whether_Remake,Whether_Franchise,Genre,New_Actor,New_Director,New_Music_Director,Lead_Star,Director,Music_Director,Number_of_Screens,Revenue(INR),Budget(INR)
0,1,0,0,12,1,0,0,260,710,103,5,5000000,85000
1,0,0,0,5,1,0,1,282,754,49,75,15000000,825000
2,0,0,0,13,0,0,0,328,689,461,525,75000000,56700000
3,0,0,0,5,1,0,0,0,303,54,800,210000000,4500000
4,0,0,0,1,1,1,1,1,4,108,1,1000000,1075000


## 7. Feature Scaling

In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 8. Train KNN Recommendation Model

In [9]:
knn = NearestNeighbors(
    n_neighbors=6,
    metric="cosine"
)
knn.fit(X_scaled)
print("Model Trained Successfully!")


Model Trained Successfully!


## 9. Recommendation Function

In [10]:
def recommend(movie_name):
    if movie_name not in movie_names.values:
        return "Movie not found."

    idx = movie_names[movie_names == movie_name].index[0]

    distances, indices = knn.kneighbors([X_scaled[idx]])

    recommendations = []

    for i in indices[0][1:]:
        recommendations.append(movie_names.iloc[i])

    return recommendations


## 10. Test Recommendation

In [11]:
movie = input("Enter Movie Name: ")

result = recommend(movie)

print("\nRecommended Movies:")

if isinstance(result, list):
    for i, movie in enumerate(result, 1):
        print(f"{i}. {movie}")
else:
    print(result)



Recommended Movies:
Movie not found.


## 11. Save Model

In [13]:
model_data = {
    "model": knn,
    "scaler": scaler,
    "movie_names": movie_names,
    "feature_columns": feature_cols,
    "encoders": encoders,
    "X_scaled": X_scaled,
    "df": df
}

joblib.dump(model_data, "knn.pkl")

['knn.pkl']